# Credit Risk Feature Engineering with LLMs

This notebook shows how to use `skfeaturellm` to automatically generate meaningful features for credit risk prediction using Large Language Models (LLMs).

## Overview

1. Load the Bank Loan dataset from Kaggle via `kagglehub`
2. Train a baseline XGBoost model to establish a benchmark
3. Use `LLMFeatureEngineer` to generate new features guided by dataset statistics
4. Evaluate features and select the most impactful ones
5. Use `fit_selective()` for automated iterative generate → select → feedback rounds
6. Export selected features to a `FeatureEngineeringTransformer` and plug it into a sklearn `Pipeline`
7. Serialize the transformer to JSON for deployment — no LLM call required

In [1]:
import os

import kagglehub
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from skfeaturellm import FeatureEngineeringTransformer, LLMFeatureEngineer

## Dataset: Bank Loan Credit Risk

The [Bank Loan Dataset](https://www.kaggle.com/datasets/udaymalviya/bank-loan-data) contains financial and demographic information about loan applicants. The prediction task is binary classification: will the loan be repaid (`loan_status = 1`) or default (`loan_status = 0`)? We use only numeric features for this tutorial.

In [2]:
path = kagglehub.dataset_download("udaymalviya/bank-loan-data")
print(f"Dataset path: {path}")

Dataset path: /Users/roberto/.cache/kagglehub/datasets/udaymalviya/bank-loan-data/versions/1


In [3]:
df = pd.read_csv(os.path.join(path, "loan_data.csv"))

df = df.astype({col: 'category' for col in df.select_dtypes(include=['object']).columns})

# exclude categorical features
df = df.select_dtypes(exclude=['object'])

print(f"Shape: {df.shape}")
print(f"\nTarget distribution:\n{df['loan_status'].value_counts().to_string()}")

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")
df.head()

Shape: (45000, 14)

Target distribution:
loan_status
0    35000
1    10000

Train: (36000, 13)  |  Test: (9000, 13)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


## Baseline Model

We train XGBoost on the raw features to establish a performance benchmark before any feature engineering.

In [4]:
baseline_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,
)
baseline_model.fit(X_train, y_train)

baseline_proba = baseline_model.predict_proba(X_test)[:, 1]
baseline_preds = baseline_model.predict(X_test)

baseline_roc_auc = roc_auc_score(y_test, baseline_proba)
baseline_avg_prec_score = average_precision_score(y_test, baseline_proba)

print("Baseline XGBoost Performance:")
print(f"  ROC AUC:  {baseline_roc_auc:.4f}")
print(f"  Avg Prec: {baseline_avg_prec_score:.4f}")

Baseline XGBoost Performance:
  ROC AUC:  0.9722
  Avg Prec: 0.9247


## LLM-Powered Feature Engineering

`LLMFeatureEngineer` uses the LLM to propose new features informed by the dataset statistics computed during `fit()`. We fit only on the training set to prevent data leakage into the test evaluation.

### Steps:
1. Define human-readable `feature_descriptions` and a `target_description`
2. Call `fit(X_train, y_train, ...)` — the LLM receives the statistics and proposes features
3. Inspect the generated ideas with `generated_features_ideas`
4. Score each feature with `evaluate_features()`
5. Apply to train and test with `transform()`

In [5]:
feature_descriptions = [
    {'name': 'person_age', 'description': 'age of the applicant in years', 'type': 'int'},
    {'name': 'person_gender', 'description': 'gender of the applicant (male, female)', 'type': 'categorical'},
    {'name': 'person_education', 'description': 'educational background of the applicant (High School, Bachelor, Master, etc.)', 'type': 'categorical'},
    {'name': 'person_income', 'description': 'annual income of the applicant in USD', 'type': 'float'},
    {'name': 'person_emp_exp', 'description': 'years of employment experience', 'type': 'float'},
    {'name': 'person_home_ownership', 'description': 'type of home ownership (RENT, OWN, MORTGAGE)', 'type': 'categorical'},
    {'name': 'loan_amnt', 'description': 'loan amount requested in USD', 'type': 'float'},
    {'name': 'loan_intent', 'description': 'purpose of the loan (PERSONAL, EDUCATION, MEDICAL, etc.)', 'type': 'categorical'},
    {'name': 'loan_int_rate', 'description': 'interest rate on the loan as a percentage', 'type': 'float'},
    {'name': 'loan_percent_income', 'description': 'ratio of loan amount to applicant income', 'type': 'float'},
    {'name': 'cb_person_cred_hist_length', 'description': 'length of the applicant credit history in years', 'type': 'float'},
    {'name': 'credit_score', 'description': 'credit score of the applicant', 'type': 'int'},
    {'name': 'previous_loan_defaults_on_file', 'description': 'whether the applicant has previous loan defaults (Yes or No)', 'type': 'categorical'}
]

target_description = 'The target variable is loan_status: 1 if the loan was successfully repaid, 0 if the applicant defaulted.'

In [6]:
engineer = LLMFeatureEngineer(
    model_name="gpt-5",
    problem_type="classification",
    feature_prefix="feng_",
    max_features=15,
)

In [7]:
engineer.fit(
    X=X_train,
    y=y_train,
    feature_descriptions=feature_descriptions,
    target_description=target_description,
)

/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...None, bin_edges=None))]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


LLMFeatureEngineer(feature_prefix='feng_', max_features=15, model_name='gpt-5',
                   problem_type=<ProblemType.CLASSIFICATION: 'classification'>)

In [9]:
print(f"Generated {len(engineer.generated_features_ideas)} feature(s):\n")
for idea in engineer.generated_features_ideas:
    print(f"  [{idea.type:>6}]  {idea.feature_name}")
    print(f"            {idea.description}\n")

Generated 15 feature(s):

  [ log1p]  log1p_person_income
            Natural log of (1 + person_income) to reduce extreme right skew (skewness ~36) and capture diminishing returns of higher income on repayment likelihood.

  [ log1p]  log1p_loan_amnt
            Natural log of (1 + loan_amnt) to reduce right skew and help the model learn proportional effects of loan size on risk.

  [ log1p]  log1p_person_emp_exp
            Log(1 + years of employment experience) to smooth non-linear effects and handle zeros while emphasizing early years of experience.

  [ log1p]  log1p_cb_person_cred_hist_length
            Log(1 + credit history length) to model diminishing marginal benefit of longer histories and reduce skew.

  [ log1p]  log1p_loan_percent_income
            Log(1 + loan_percent_income) to stabilize variance and model non-linear effects of debt-to-income burden (noting higher values among defaulters).

  [   mul]  rate_x_dti
            Interaction between loan interest rate and

## Results: Baseline vs. Engineered Features

We apply the engineered features to both train and test sets, retrain XGBoost, and compare against the baseline.

In [10]:
X_train_eng = engineer.transform(X_train)
X_test_eng = engineer.transform(X_test)

obj_cols = X_train_eng.select_dtypes(include=['object']).columns
X_train_eng[obj_cols] = X_train_eng[obj_cols].astype('category')
X_test_eng[obj_cols] = X_test_eng[obj_cols].astype('category')

results = {}

for idea in engineer.generated_features_ideas:

    feature_to_add = 'feng_' + idea.feature_name

    features = X_train.columns.tolist() + [feature_to_add]
    
    eng_model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )
    eng_model.fit(X_train_eng[features], y_train)

    eng_proba = eng_model.predict_proba(X_test_eng[features])[:, 1]
    eng_preds = eng_model.predict(X_test_eng[features])

    eng_roc_auc = roc_auc_score(y_test, eng_proba)
    eng_avg_prec_score = average_precision_score(y_test, eng_proba)

    eng_roc_auc_delta = 100 * (eng_roc_auc - baseline_roc_auc)
    eng_avg_prec_delta = 100 * (eng_avg_prec_score - baseline_avg_prec_score)

    results[idea.feature_name] = {
        "ROC AUC": eng_roc_auc,
        "Avg Prec": eng_avg_prec_score,
        "ROC AUC Δ (%)": eng_roc_auc_delta,
        "Avg Prec Δ (%)": eng_avg_prec_delta,
    }


df_results = pd.DataFrame(results).T.sort_values(by="ROC AUC Δ (%)", ascending=False).rename_axis("Feature added")

df_results

,ROC AUC,Avg Prec,ROC AUC Δ (%),Avg Prec Δ (%)
Feature added,,,,
loan_per_credit_year,0.972670,0.926050,0.050161,0.138568
dti_times_age,0.972547,0.925698,0.037907,0.103338
income_per_credit_year,0.972450,0.925327,0.028157,0.066271
income_minus_loan_amnt,0.972405,0.925377,0.023682,0.071321
log1p_person_income,0.972168,0.924664,0.000000,0.000000
log1p_loan_amnt,0.972168,0.924664,0.000000,0.000000
log1p_person_emp_exp,0.972168,0.924664,0.000000,0.000000
log1p_cb_person_cred_hist_length,0.972168,0.924664,0.000000,0.000000
log1p_loan_percent_income,0.972168,0.924664,0.000000,0.000000


In [11]:
features_to_add = [f'feng_{feature}' for feature in df_results[df_results["ROC AUC Δ (%)"] > 0].index.to_list()]

features = X_train.columns.tolist() + features_to_add

final_eng_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,
)
final_eng_model.fit(X_train_eng[features], y_train)

final_eng_proba = final_eng_model.predict_proba(X_test_eng[features])[:, 1]
final_eng_preds = final_eng_model.predict(X_test_eng[features])

final_eng_roc_auc = roc_auc_score(y_test, final_eng_proba)
final_eng_avg_prec_score = average_precision_score(y_test, final_eng_proba)

final_eng_roc_auc_delta = 100 * (final_eng_roc_auc - baseline_roc_auc)
final_eng_avg_prec_delta = 100 * (final_eng_avg_prec_score - baseline_avg_prec_score)

results = pd.DataFrame(
    {
        "Model": ["XGBoost Baseline", "XGBoost + LLM Features"],
        "ROC AUC": [baseline_roc_auc, final_eng_roc_auc],
        "Avg Prec": [baseline_avg_prec_score, final_eng_avg_prec_score],
    }
).set_index("Model")


results


,ROC AUC,Avg Prec
Model,,
XGBoost Baseline,0.972168,0.924664
XGBoost + LLM Features,0.972987,0.926750


## Iterative Feature Engineering with `fit_selective()`

`fit_selective()` automates the **generate → select → feedback** loop over multiple rounds. In each round the LLM proposes new features, a scikit-learn-compatible selector scores them **in the context of all original features**, and the results are fed back to the LLM for the next round. Only features that survive selection are kept.

**When to use it:**
- You want the LLM to progressively focus its proposals based on what actually helps the model
- You have a validation set and want unbiased, generalisation-aware feature selection

**Steps:**
1. Split training data into a train / validation split to avoid leakage into selection
2. Initialise a scikit-learn selector (e.g. `SelectKBest`, `SelectFromModel`)
3. Call `fit_selective()` — rounds run automatically, history is managed internally
4. Call `transform()` to apply the surviving features

In [9]:
from sklearn.feature_selection import SelectFromModel

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=0, stratify=y_train
)

engineer_sel = LLMFeatureEngineer(
    model_name="gpt-5",
    problem_type="classification",
    feature_prefix="feng_sel_",
    max_features=10,
    verbose=2,
)

selector = SelectFromModel(
    estimator=XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True
    ),
    importance_getter="feature_importances_",
    prefit=False
)

engineer_sel.fit_selective(
    X=X_tr,
    y=y_tr,
    selector=selector,
    n_rounds=5,
    eval_set=(X_val, y_val),
    feature_descriptions=feature_descriptions,
    target_description=target_description,
)

[fit_selective] Round 1/5
  Querying LLM...
  Generated 10 idea(s)
  Applying transformations...
  Created 10/10 feature(s). Running selector on 23 feature(s) (13 original + 10 new)...
  Selected: 2 | Rejected: 8
    + feng_sel_interest_stress_index
    + feng_sel_rate_over_credit_score
    - feng_sel_log1p_person_income
    - feng_sel_log1p_loan_amnt
    - feng_sel_sqrt_loan_percent_income
    - feng_sel_credit_history_to_age
    - feng_sel_experience_to_age
    - feng_sel_loan_to_credit_score
    - feng_sel_log1p_cb_person_cred_hist_length
    - feng_sel_min_credit_score_650
[fit_selective] Round 2/5
  Querying LLM...


/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...None, bin_edges=None))]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


  Generated 10 idea(s)
  Applying transformations...
  Created 10/10 feature(s). Running selector on 23 feature(s) (13 original + 10 new)...
  Selected: 2 | Rejected: 8
    + feng_sel_affordability_vs_pricing_gap
    + feng_sel_income_minus_loan
    - feng_sel_rate_x_amount
    - feng_sel_rate_over_credit_history
    - feng_sel_rate_x_credit_history
    - feng_sel_rate_over_age
    - feng_sel_rate_x_experience
    - feng_sel_amount_per_credit_year
    - feng_sel_age_at_first_credit
    - feng_sel_rate_squared
[fit_selective] Round 3/5
  Querying LLM...


/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...None, bin_edges=None))]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


  Generated 10 idea(s)
  Applying transformations...
  Created 9/10 feature(s). Running selector on 22 feature(s) (13 original + 9 new)...
  Selected: 2 | Rejected: 8
    + feng_sel_pricing_plus_affordability_sum
    + feng_sel_income_per_rate
    - feng_sel_affordability_per_pricing_ratio
    - feng_sel_rate_minus_11pct
    - feng_sel_income_share_minus_0_15
    - feng_sel_income_per_credit_point
    - feng_sel_credit_score_per_credit_year
    - feng_sel_income_minus_rate
    - feng_sel_rate_floor_12pct
    - feng_sel_score_over_income_share
[fit_selective] Round 4/5
  Querying LLM...


...

  Generated 10 idea(s)
  Applying transformations...
  Created 10/10 feature(s). Running selector on 23 feature(s) (13 original + 10 new)...
  Selected: 0 | Rejected: 10
    - feng_sel_amount_per_rate
    - feng_sel_credit_score_minus_rate
    - feng_sel_score_x_income_share
    - feng_sel_amount_x_income_share
    - feng_sel_income_plus_loan
    - feng_sel_income_over_age
    - feng_sel_loan_amount_over_age
    - feng_sel_income_share_x_age
    - feng_sel_income_share_x_experience
    - feng_sel_credit_score_per_age
[fit_selective] Round 5/5
  Querying LLM...


/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...ge'], parameters=None)]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


  Generated 8 idea(s)
  Applying transformations...
  Created 8/8 feature(s). Running selector on 21 feature(s) (13 original + 8 new)...
  Selected: 1 | Rejected: 7
    + feng_sel_income_over_loan_amount
    - feng_sel_residual_income_ratio
    - feng_sel_max_pricing_or_burden
    - feng_sel_min_pricing_or_burden
    - feng_sel_experience_over_amount
    - feng_sel_credit_history_over_amount
    - feng_sel_min_income_or_loan
    - feng_sel_max_income_or_loan
[fit_selective] Done. Total selected features: 7


/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...nt'], parameters=None)]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


LLMFeatureEngineer(feature_prefix='feng_sel_', max_features=10,
                   model_name='gpt-5',
                   problem_type=<ProblemType.CLASSIFICATION: 'classification'>,
                   verbose=2)

In [10]:
engineer_sel.generated_features_ideas

[FeatureEngineeringIdea(type='mul', feature_name='interest_stress_index', description='Interaction of pricing and affordability: higher interest rates amplify the burden of a high loan share of income.', columns=['loan_int_rate', 'loan_percent_income'], parameters=None),
 FeatureEngineeringIdea(type='div', feature_name='rate_over_credit_score', description='Interest rate normalized by credit score; higher values indicate pricing that is high relative to applicant creditworthiness, potentially signaling elevated risk.', columns=['loan_int_rate', 'credit_score'], parameters=None),
 FeatureEngineeringIdea(type='sub', feature_name='affordability_vs_pricing_gap', description='Difference between interest rate and loan share of income; aggregates two risk drivers (pricing and affordability) on a comparable scale.', columns=['loan_int_rate', 'loan_percent_income'], parameters=None),
 FeatureEngineeringIdea(type='sub', feature_name='income_minus_loan', description='Residual annual income after 

In [7]:
X_train_sel = engineer_sel.transform(X_train)
X_test_sel = engineer_sel.transform(X_test)

X_train_sel

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,feng_sel_interest_income_pressure,feng_sel_rate_per_credit_point,feng_sel_leverage_per_credit_point
6048,24.0,male,Master,58914.0,2,OWN,4400.0,VENTURE,5.99,0.07,4.0,656,Yes,0.4193,0.009131,0.000107
3346,23.0,female,High School,45873.0,2,RENT,11000.0,VENTURE,11.01,0.24,2.0,634,Yes,2.6424,0.017366,0.000379
17998,29.0,female,Master,240947.0,7,MORTGAGE,10000.0,VENTURE,12.69,0.04,9.0,638,Yes,0.5076,0.019890,0.000063
24988,30.0,female,Bachelor,96316.0,10,MORTGAGE,6000.0,MEDICAL,13.49,0.06,8.0,682,No,0.8094,0.019780,0.000088
23231,29.0,male,Bachelor,73033.0,7,MORTGAGE,8000.0,PERSONAL,10.51,0.11,8.0,644,Yes,1.1561,0.016320,0.000171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3245,23.0,male,Bachelor,89921.0,3,RENT,3500.0,EDUCATION,12.87,0.04,3.0,634,Yes,0.5148,0.020300,0.000063
17803,25.0,female,Associate,35911.0,2,RENT,3000.0,EDUCATION,6.62,0.08,4.0,573,No,0.5296,0.011553,0.000140
28842,30.0,male,Bachelor,72786.0,9,OWN,20000.0,EDUCATION,6.17,0.27,9.0,477,Yes,1.6659,0.012935,0.000566
21553,30.0,male,Associate,39629.0,8,RENT,6000.0,MEDICAL,12.99,0.15,9.0,689,No,1.9485,0.018853,0.000218


In [11]:
X_train_sel = engineer_sel.transform(X_train)
X_test_sel = engineer_sel.transform(X_test)

obj_cols = X_train_sel.select_dtypes(include=["object"]).columns
X_train_sel[obj_cols] = X_train_sel[obj_cols].astype("category")
X_test_sel[obj_cols] = X_test_sel[obj_cols].astype("category")

# generated_features is populated by transform() — features that were successfully created
sel_new_cols = [f"feng_sel_{idea.feature_name}" for idea in engineer_sel.generated_features]
sel_features = X_train.columns.tolist() + sel_new_cols

sel_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,
)
sel_model.fit(X_train_sel[sel_features], y_train)

sel_proba = sel_model.predict_proba(X_test_sel[sel_features])[:, 1]
sel_roc_auc = roc_auc_score(y_test, sel_proba)
sel_avg_prec = average_precision_score(y_test, sel_proba)

pd.DataFrame(
    {
        "Model": ["XGBoost Baseline", "XGBoost + fit_selective Features"],
        "ROC AUC": [baseline_roc_auc, sel_roc_auc],
        "Avg Prec": [baseline_avg_prec_score, sel_avg_prec],
    }
).set_index("Model")

,ROC AUC,Avg Prec
Model,,
XGBoost Baseline,0.972168,0.924664
XGBoost + fit_selective Features,0.972627,0.926393


## Production: FeatureEngineeringTransformer + sklearn Pipeline

`LLMFeatureEngineer` is designed for **exploration** — it calls the LLM during `fit()`. For production, convert the selected features into a `FeatureEngineeringTransformer`: a fully deterministic scikit-learn transformer with no LLM dependency.

### Steps:
1. Call `to_transformer(features=[...])` to export only the beneficial features
2. Plug the transformer directly into a sklearn `Pipeline`
3. Call `save()` to serialize the configs to JSON — no LLM needed on reload

In [13]:
# Export only the features that improved the baseline
transformer = engineer.to_transformer(features=features_to_add)

print(f"Exported {len(transformer.transformations)} transformation(s) to FeatureEngineeringTransformer:")
for t in transformer.transformations:
    print(f"  [{t['type']:>6}]  {t['feature_name']}")

Exported 3 transformation(s) to FeatureEngineeringTransformer:
  [   mul]  feng_rate_weighted_loan_amount
  [   div]  feng_income_per_loan_dollar
  [   div]  feng_credit_score_per_rate


In [14]:
# Build a sklearn Pipeline: FeatureEngineeringTransformer + XGBoost
pipeline = Pipeline([
    ("features", transformer),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )),
])

pipeline.fit(X_train, y_train)

pipeline_proba = pipeline.predict_proba(X_test)[:, 1]
pipeline_roc_auc = roc_auc_score(y_test, pipeline_proba)
pipeline_avg_prec = average_precision_score(y_test, pipeline_proba)

print("Pipeline (FeatureEngineeringTransformer + XGBoost):")
print(f"  ROC AUC:  {pipeline_roc_auc:.4f}  (baseline: {baseline_roc_auc:.4f})")
print(f"  Avg Prec: {pipeline_avg_prec:.4f}  (baseline: {baseline_avg_prec_score:.4f})")

Pipeline (FeatureEngineeringTransformer + XGBoost):
  ROC AUC:  0.9728  (baseline: 0.9722)
  Avg Prec: 0.9266  (baseline: 0.9247)


### Serializing for Deployment

`FeatureEngineeringTransformer.save()` writes only the transformation configs to JSON — no LLM, no fitted state. On reload, call `fit()` (or let the `Pipeline` call it) to re-learn any stateful parameters such as bin edges.

In [15]:
# Save transformation configs to JSON (no LLM call stored)
transformer.save("transformer.json")
print("Saved transformer.json")

# Reload and rebuild the pipeline — no LLM needed
loaded_transformer = FeatureEngineeringTransformer.load("transformer.json")

pipeline_loaded = Pipeline([
    ("features", loaded_transformer),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )),
])

pipeline_loaded.fit(X_train, y_train)
loaded_roc_auc = roc_auc_score(y_test, pipeline_loaded.predict_proba(X_test)[:, 1])
print(f"Loaded pipeline ROC AUC: {loaded_roc_auc:.4f}")

Saved transformer.json
Loaded pipeline ROC AUC: 0.9728
